## ✅ Step 1: Start Spark Session

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("MLlib_Classification_Basics")
    .master("local[*]")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/16 22:19:08 WARN Utils: Your hostname, bhuvaneshwaran-Latitude-5420, resolves to a loopback address: 127.0.1.1; using 10.202.133.134 instead (on interface wlp0s20f3)
26/02/16 22:19:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/16 22:19:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
spark

### ✅ .appName("MLlib_Classification_Basics")

    Names your app (shows in Spark UI)

### ✅ .master("local[*]")

    Runs Spark locally

    * means use all CPU cores

### ✅ .getOrCreate()

    Creates a session if none exists
    
    Otherwise reuses existing session

## ✅ Step 2: Read a parquet file

In [3]:
df = spark.read.parquet('/home/bhuvaneshwaran/Desktop/Medium/parquetFiles/Emp_layoff.parquet')
df.show(5)

+---------+---+----------+--------------------+------+-----------+------------+------+------+----+---------------+-------------+------------------+-----+----------+---+------------------+--------------+--------------+
|   Emp_ID|Age|Experience|              Domain|Salary|Skill_Score|Domain_Score|Layoff|Python|Java|Cloud Computing|Generative AI|Prompt Engineering|React|Kubernetes|SQL|Automation Testing|Manual Testing|Legacy Systems|
+---------+---+----------+--------------------+------+-----------+------------+------+------+----+---------------+-------------+------------------+-----+----------+---+------------------+--------------+--------------+
|EMP100000| 48|        26|Software Development|154517|      0.712|        0.75|     0|     0|   1|              0|            0|                 1|    0|         1|  0|                 0|             0|             1|
|EMP100001| 45|        22|   Technical Support|107297|      0.792|         0.4|     1|     1|   1|              0|            1|

In [4]:
df.columns

['Emp_ID',
 'Age',
 'Experience',
 'Domain',
 'Salary',
 'Skill_Score',
 'Domain_Score',
 'Layoff',
 'Python',
 'Java',
 'Cloud Computing',
 'Generative AI',
 'Prompt Engineering',
 'React',
 'Kubernetes',
 'SQL',
 'Automation Testing',
 'Manual Testing',
 'Legacy Systems']

In [5]:
df.groupby("Layoff").count().show()

+------+-----+
|Layoff|count|
+------+-----+
|     0|27295|
|     1|22705|
+------+-----+



✅ Layoff

    This is the target column

    Must be numeric (0/1) for classification in Spark

In [6]:
df.printSchema()

root
 |-- Emp_ID: string (nullable = true)
 |-- Age: long (nullable = true)
 |-- Experience: long (nullable = true)
 |-- Domain: string (nullable = true)
 |-- Salary: long (nullable = true)
 |-- Skill_Score: double (nullable = true)
 |-- Domain_Score: double (nullable = true)
 |-- Layoff: long (nullable = true)
 |-- Python: long (nullable = true)
 |-- Java: long (nullable = true)
 |-- Cloud Computing: long (nullable = true)
 |-- Generative AI: long (nullable = true)
 |-- Prompt Engineering: long (nullable = true)
 |-- React: long (nullable = true)
 |-- Kubernetes: long (nullable = true)
 |-- SQL: long (nullable = true)
 |-- Automation Testing: long (nullable = true)
 |-- Manual Testing: long (nullable = true)
 |-- Legacy Systems: long (nullable = true)



In [7]:
df.select("Domain","Emp_ID").show()

+--------------------+---------+
|              Domain|   Emp_ID|
+--------------------+---------+
|Software Development|EMP100000|
|   Technical Support|EMP100001|
|              DevOps|EMP100002|
|   Technical Support|EMP100003|
|  Project Management|EMP100004|
|  Project Management|EMP100005|
|   Quality Assurance|EMP100006|
|       Data Analysis|EMP100007|
|   Quality Assurance|EMP100008|
|  Project Management|EMP100009|
|   Technical Support|EMP100010|
|              DevOps|EMP100011|
|Software Development|EMP100012|
|  Project Management|EMP100013|
|Software Development|EMP100014|
|        UI/UX Design|EMP100015|
|              DevOps|EMP100016|
|   Technical Support|EMP100017|
|        UI/UX Design|EMP100018|
|Software Development|EMP100019|
+--------------------+---------+
only showing top 20 rows


In [8]:
# from pyspark.sql.functions import col

In [9]:
df.groupBy("Domain").count().show()

+--------------------+-----+
|              Domain|count|
+--------------------+-----+
|Software Development| 7231|
|       Data Analysis| 7106|
|   Quality Assurance| 7109|
|   Technical Support| 7111|
|        UI/UX Design| 7105|
|  Project Management| 7083|
|              DevOps| 7255|
+--------------------+-----+



In [10]:
df.columns

['Emp_ID',
 'Age',
 'Experience',
 'Domain',
 'Salary',
 'Skill_Score',
 'Domain_Score',
 'Layoff',
 'Python',
 'Java',
 'Cloud Computing',
 'Generative AI',
 'Prompt Engineering',
 'React',
 'Kubernetes',
 'SQL',
 'Automation Testing',
 'Manual Testing',
 'Legacy Systems']

## ✅ Step 3: Convert Features Into a Vector (VectorAssembler)

Spark ML models do NOT accept multiple feature columns directly.

They want ONE column called:
👉 features (vector format)

So we use:

### ✔ Proper Spark Way to One-Hot Encode
## Step 1 — Convert category → index

In [11]:
from pyspark.ml.feature import StringIndexer

indexer = StringIndexer(
    inputCol="Domain",
    outputCol="Domain_index"
)

In [12]:
df_indexed = indexer.fit(df).transform(df)
df_indexed.show()

+---------+---+----------+--------------------+------+-----------+------------+------+------+----+---------------+-------------+------------------+-----+----------+---+------------------+--------------+--------------+------------+
|   Emp_ID|Age|Experience|              Domain|Salary|Skill_Score|Domain_Score|Layoff|Python|Java|Cloud Computing|Generative AI|Prompt Engineering|React|Kubernetes|SQL|Automation Testing|Manual Testing|Legacy Systems|Domain_index|
+---------+---+----------+--------------------+------+-----------+------------+------+------+----+---------------+-------------+------------------+-----+----------+---+------------------+--------------+--------------+------------+
|EMP100000| 48|        26|Software Development|154517|      0.712|        0.75|     0|     0|   1|              0|            0|                 1|    0|         1|  0|                 0|             0|             1|         1.0|
|EMP100001| 45|        22|   Technical Support|107297|      0.792|         0

## Step 2 — One Hot Encode

In [13]:
from pyspark.ml.feature import OneHotEncoder

encoder = OneHotEncoder(
    inputCols=["Domain_index"],
    outputCols=["Domain_vec"]
)

df_encoded = encoder.fit(df_indexed).transform(df_indexed)
df_encoded.show()

+---------+---+----------+--------------------+------+-----------+------------+------+------+----+---------------+-------------+------------------+-----+----------+---+------------------+--------------+--------------+------------+-------------+
|   Emp_ID|Age|Experience|              Domain|Salary|Skill_Score|Domain_Score|Layoff|Python|Java|Cloud Computing|Generative AI|Prompt Engineering|React|Kubernetes|SQL|Automation Testing|Manual Testing|Legacy Systems|Domain_index|   Domain_vec|
+---------+---+----------+--------------------+------+-----------+------------+------+------+----+---------------+-------------+------------------+-----+----------+---+------------------+--------------+--------------+------------+-------------+
|EMP100000| 48|        26|Software Development|154517|      0.712|        0.75|     0|     0|   1|              0|            0|                 1|    0|         1|  0|                 0|             0|             1|         1.0|(6,[1],[1.0])|
|EMP100001| 45|     

In [14]:
df_encoded.select('Domain','Domain_index','Domain_vec').show()

+--------------------+------------+-------------+
|              Domain|Domain_index|   Domain_vec|
+--------------------+------------+-------------+
|Software Development|         1.0|(6,[1],[1.0])|
|   Technical Support|         2.0|(6,[2],[1.0])|
|              DevOps|         0.0|(6,[0],[1.0])|
|   Technical Support|         2.0|(6,[2],[1.0])|
|  Project Management|         6.0|    (6,[],[])|
|  Project Management|         6.0|    (6,[],[])|
|   Quality Assurance|         3.0|(6,[3],[1.0])|
|       Data Analysis|         4.0|(6,[4],[1.0])|
|   Quality Assurance|         3.0|(6,[3],[1.0])|
|  Project Management|         6.0|    (6,[],[])|
|   Technical Support|         2.0|(6,[2],[1.0])|
|              DevOps|         0.0|(6,[0],[1.0])|
|Software Development|         1.0|(6,[1],[1.0])|
|  Project Management|         6.0|    (6,[],[])|
|Software Development|         1.0|(6,[1],[1.0])|
|        UI/UX Design|         5.0|(6,[5],[1.0])|
|              DevOps|         0.0|(6,[0],[1.0])|


## Step 3 — Convert vector into columns (optional but useful)

In [15]:
from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import col

df_final = df_encoded.withColumn(
    "Domain_arr",
    vector_to_array(col("Domain_vec"))
)
df_final.show()

+---------+---+----------+--------------------+------+-----------+------------+------+------+----+---------------+-------------+------------------+-----+----------+---+------------------+--------------+--------------+------------+-------------+--------------------+
|   Emp_ID|Age|Experience|              Domain|Salary|Skill_Score|Domain_Score|Layoff|Python|Java|Cloud Computing|Generative AI|Prompt Engineering|React|Kubernetes|SQL|Automation Testing|Manual Testing|Legacy Systems|Domain_index|   Domain_vec|          Domain_arr|
+---------+---+----------+--------------------+------+-----------+------------+------+------+----+---------------+-------------+------------------+-----+----------+---+------------------+--------------+--------------+------------+-------------+--------------------+
|EMP100000| 48|        26|Software Development|154517|      0.712|        0.75|     0|     0|   1|              0|            0|                 1|    0|         1|  0|                 0|             0|

In [16]:
df_final.select('Domain','Domain_index','Domain_vec','Domain_arr').show()

+--------------------+------------+-------------+--------------------+
|              Domain|Domain_index|   Domain_vec|          Domain_arr|
+--------------------+------------+-------------+--------------------+
|Software Development|         1.0|(6,[1],[1.0])|[0.0, 1.0, 0.0, 0...|
|   Technical Support|         2.0|(6,[2],[1.0])|[0.0, 0.0, 1.0, 0...|
|              DevOps|         0.0|(6,[0],[1.0])|[1.0, 0.0, 0.0, 0...|
|   Technical Support|         2.0|(6,[2],[1.0])|[0.0, 0.0, 1.0, 0...|
|  Project Management|         6.0|    (6,[],[])|[0.0, 0.0, 0.0, 0...|
|  Project Management|         6.0|    (6,[],[])|[0.0, 0.0, 0.0, 0...|
|   Quality Assurance|         3.0|(6,[3],[1.0])|[0.0, 0.0, 0.0, 1...|
|       Data Analysis|         4.0|(6,[4],[1.0])|[0.0, 0.0, 0.0, 0...|
|   Quality Assurance|         3.0|(6,[3],[1.0])|[0.0, 0.0, 0.0, 1...|
|  Project Management|         6.0|    (6,[],[])|[0.0, 0.0, 0.0, 0...|
|   Technical Support|         2.0|(6,[2],[1.0])|[0.0, 0.0, 1.0, 0...|
|     

In [17]:
df_final.printSchema()

root
 |-- Emp_ID: string (nullable = true)
 |-- Age: long (nullable = true)
 |-- Experience: long (nullable = true)
 |-- Domain: string (nullable = true)
 |-- Salary: long (nullable = true)
 |-- Skill_Score: double (nullable = true)
 |-- Domain_Score: double (nullable = true)
 |-- Layoff: long (nullable = true)
 |-- Python: long (nullable = true)
 |-- Java: long (nullable = true)
 |-- Cloud Computing: long (nullable = true)
 |-- Generative AI: long (nullable = true)
 |-- Prompt Engineering: long (nullable = true)
 |-- React: long (nullable = true)
 |-- Kubernetes: long (nullable = true)
 |-- SQL: long (nullable = true)
 |-- Automation Testing: long (nullable = true)
 |-- Manual Testing: long (nullable = true)
 |-- Legacy Systems: long (nullable = true)
 |-- Domain_index: double (nullable = false)
 |-- Domain_vec: vector (nullable = true)
 |-- Domain_arr: array (nullable = false)
 |    |-- element: double (containsNull = false)



| Column       | Purpose                    | Can be used in ML?      |
| ------------ | -------------------------- | ----------------------- |
| Domain       | Raw string                 | ❌ No                    |
| Domain_index | Numeric label ID           | ❌ Dangerous             |
| Domain_vec   | One-Hot encoded vector     | ✅ YES (correct feature) |
| Domain_arr   | Expanded array (debugging) | ❌ No                    |


In [18]:
df_final.columns

['Emp_ID',
 'Age',
 'Experience',
 'Domain',
 'Salary',
 'Skill_Score',
 'Domain_Score',
 'Layoff',
 'Python',
 'Java',
 'Cloud Computing',
 'Generative AI',
 'Prompt Engineering',
 'React',
 'Kubernetes',
 'SQL',
 'Automation Testing',
 'Manual Testing',
 'Legacy Systems',
 'Domain_index',
 'Domain_vec',
 'Domain_arr']

In [19]:
exc_cols=["Emp_ID",'Domain','Domain_index','Domain_arr','Domain']
feature_cols = [c for c in df_final.columns if c not in exc_cols]
feature_cols

['Age',
 'Experience',
 'Salary',
 'Skill_Score',
 'Domain_Score',
 'Layoff',
 'Python',
 'Java',
 'Cloud Computing',
 'Generative AI',
 'Prompt Engineering',
 'React',
 'Kubernetes',
 'SQL',
 'Automation Testing',
 'Manual Testing',
 'Legacy Systems',
 'Domain_vec']

In [20]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

df_features = assembler.transform(df_final.drop("Emp_ID",'Domain','Domain_index','Domain_arr'))
df_features.select("features", "Layoff").show(truncate=False)


+---------------------------------------------------------------------------------------------------+------+
|features                                                                                           |Layoff|
+---------------------------------------------------------------------------------------------------+------+
|(23,[0,1,2,3,4,7,10,12,16,18],[48.0,26.0,154517.0,0.712,0.75,1.0,1.0,1.0,1.0,1.0])                 |0     |
|(23,[0,1,2,3,4,5,6,7,9,12,13,15,19],[45.0,22.0,107297.0,0.792,0.4,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])|1     |
|(23,[0,1,2,3,4,7,9,14,15,17],[48.0,27.0,159245.0,0.662,0.95,1.0,1.0,1.0,1.0,1.0])                  |0     |
|(23,[0,1,2,3,4,6,8,10,12,13,14,19],[27.0,5.0,71130.0,0.875,0.4,1.0,1.0,1.0,1.0,1.0,1.0,1.0])       |0     |
|(23,[0,1,2,3,4,5,9,10,15,16],[30.0,9.0,95419.0,0.612,0.6,1.0,1.0,1.0,1.0,1.0])                     |1     |
|(23,[0,1,2,3,4,5,6,11,15],[52.0,30.0,147211.0,0.7,0.6,1.0,1.0,1.0,1.0])                            |1     |
|(23,[0,1,2,3,4,5,8

26/02/16 22:19:15 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


✅ inputCols=['Age','Experience','Salary','Skill_Score','Domain_Score','Layoff','Python','Java','Cloud Computing','Generative AI','Prompt Engineering','React','Kubernetes','SQL','Automation Testing','Manual Testing','Legacy Systems','Domain_vec']

    List of feature columns to combine

✅ outputCol="features"

    The new column name Spark ML will use

✅ .transform(df)

    Applies transformation to DataFrame

✅ Alternative

    If you have many features, you can generate the inputCols list dynamically:

## ✅ Step 4: Train-Test Split

In [21]:
train_df, test_df = df_features.randomSplit([0.8, 0.2], seed=42)

In [22]:
train_df.show()

+---+----------+------+-----------+------------+------+------+----+---------------+-------------+------------------+-----+----------+---+------------------+--------------+--------------+-------------+--------------------+
|Age|Experience|Salary|Skill_Score|Domain_Score|Layoff|Python|Java|Cloud Computing|Generative AI|Prompt Engineering|React|Kubernetes|SQL|Automation Testing|Manual Testing|Legacy Systems|   Domain_vec|            features|
+---+----------+------+-----------+------------+------+------+----+---------------+-------------+------------------+-----+----------+---+------------------+--------------+--------------+-------------+--------------------+
| 22|         0| 30000|      0.575|         0.4|     0|     0|   0|              0|            0|                 1|    1|         0|  0|                 0|             1|             1|(6,[2],[1.0])|(23,[0,2,3,4,10,1...|
| 22|         0| 30000|      0.617|         0.4|     0|     0|   1|              0|            0|               

✅ [0.8, 0.2]

    80% training
    
    20% testing

✅ seed=42

    Fixes randomness
    
    So results stay the same every run (important for learning)

✅ Alternative

    If dataset is small, even [0.7, 0.3] is okay.

## ✅ Step 5: Train a Classification Model (Logistic Regression)

In [23]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="Layoff",
    predictionCol="prediction",
    probabilityCol="probability",
    maxIter=50
)

model = lr.fit(train_df)


In [24]:
model

LogisticRegressionModel: uid=LogisticRegression_6ba94fe125e7, numClasses=2, numFeatures=23

✅ featuresCol="features"

    Input column for features vector
    
    Default = "features"
    
    We set it explicitly for clarity

✅ labelCol="label"

    Target column
    
    Default = "label"

✅ predictionCol="prediction"

    Output predicted class (0 or 1)

✅ probabilityCol="probability"

    Output probability scores like [p0, p1]

✅ maxIter=50

    Number of training iterations
    
    More iterations → better convergence
    
    But too high can be slower

In [25]:
predictions = model.transform(test_df)
predictions.select("age", "salary", "Layoff", "prediction", "probability","rawPrediction").show(truncate=False)

+---+------+------+----------+------------------------------------------+----------------------------------------+
|age|salary|Layoff|prediction|probability                               |rawPrediction                           |
+---+------+------+----------+------------------------------------------+----------------------------------------+
|22 |30000 |1     |1.0       |[1.0282958718868346E-8,0.9999999897170413]|[-18.392777794927195,18.392777794927195]|
|22 |30000 |0     |0.0       |[0.9999999952957415,4.704258538268391E-9] |[19.174797665965933,-19.174797665965933]|
|22 |30421 |0     |0.0       |[0.9999999932054555,6.794544482602305E-9] |[18.80714583583938,-18.80714583583938]  |
|22 |34535 |0     |0.0       |[0.9999999930566998,6.943300157047361E-9] |[18.78548866159373,-18.78548866159373]  |
|22 |37502 |0     |0.0       |[0.9999999945738915,5.4261084514450886E-9]|[19.032043638434093,-19.032043638434093]|
|22 |40004 |1     |1.0       |[1.1368590417413658E-8,0.9999999886314096]|[-18.29

✅ model.transform(test_df)

    Applies trained model to test data

Adds new columns:

    prediction
    
    probability
    
    rawPrediction

## ✅ Step 7: Evaluate the Model
## ✅ Accuracy using MulticlassClassificationEvaluator

In [26]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(
    labelCol="Layoff",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = evaluator.evaluate(predictions)
print("Accuracy:", accuracy)


Accuracy: 1.0


✅ labelCol="label"

    Actual ground truth values

✅ predictionCol="prediction"

    Model predicted values

✅ metricName="accuracy"

    Measures how many predictions are correct

Alternatives:

    "f1"
    
    "weightedPrecision"
    
    "weightedRecall"

In [27]:
lr

LogisticRegression_6ba94fe125e7

In [28]:
predictions

DataFrame[Age: bigint, Experience: bigint, Salary: bigint, Skill_Score: double, Domain_Score: double, Layoff: bigint, Python: bigint, Java: bigint, Cloud Computing: bigint, Generative AI: bigint, Prompt Engineering: bigint, React: bigint, Kubernetes: bigint, SQL: bigint, Automation Testing: bigint, Manual Testing: bigint, Legacy Systems: bigint, Domain_vec: vector, features: vector, rawPrediction: vector, probability: vector, prediction: double]

## Accuracy & F1 (Built-in Spark Evaluator)

In [29]:
accuracy = evaluator.evaluate(predictions, {evaluator.metricName: "accuracy"})
precision = evaluator.evaluate(predictions, {evaluator.metricName: "weightedPrecision"})
recall = evaluator.evaluate(predictions, {evaluator.metricName: "weightedRecall"})
f1 = evaluator.evaluate(predictions, {evaluator.metricName: "f1"})

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0


##  Confusion Matrix (Very Important for Interviews)

Spark does NOT give it directly.
We use RDD metrics:

In [31]:
from pyspark.mllib.evaluation import MulticlassMetrics

preds_and_labels = predictions.select("prediction","Layoff") \
    .rdd.map(lambda x: (float(x[0]), float(x[1])))

metrics = MulticlassMetrics(preds_and_labels)

print("Confusion Matrix:")
print(metrics.confusionMatrix().toArray())


/home/bhuvaneshwaran/Documents/Virt_Envs/pyspark_env/lib/python3.11/site-packages/pyspark/sql/context.py:157: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


Confusion Matrix:
[[5510.    0.]
 [   0. 4542.]]


|          | Pred 0 | Pred 1 |
| -------- | ------ | ------ |
| Actual 0 | TN     | FP     |
| Actual 1 | FN     | TP     |


## Individual Precision/Recall per class (advanced)

In [32]:
print("Precision class 0:", metrics.precision(0.0))
print("Precision class 1:", metrics.precision(1.0))

print("Recall class 0:", metrics.recall(0.0))
print("Recall class 1:", metrics.recall(1.0))


Precision class 0: 1.0
Precision class 1: 1.0
Recall class 0: 1.0
Recall class 1: 1.0


## ✅ Step 8: Build Everything Using a Pipeline (Best Practice)

Instead of calling assembler and model separately, we can chain them into a pipeline.

In [30]:
from pyspark.ml import Pipeline

pipeline = Pipeline(stages=[assembler, lr])
pipeline_model = pipeline.fit(df_final.drop("Emp_ID",'Domain','Domain_index','Domain_arr'))

pipeline_predictions = pipeline_model.transform(df_final.drop("Emp_ID",'Domain','Domain_index','Domain_arr'))
pipeline_predictions.select("features", "Layoff", "prediction").show()


+--------------------+------+----------+
|            features|Layoff|prediction|
+--------------------+------+----------+
|(23,[0,1,2,3,4,7,...|     0|       0.0|
|(23,[0,1,2,3,4,5,...|     1|       1.0|
|(23,[0,1,2,3,4,7,...|     0|       0.0|
|(23,[0,1,2,3,4,6,...|     0|       0.0|
|(23,[0,1,2,3,4,5,...|     1|       1.0|
|(23,[0,1,2,3,4,5,...|     1|       1.0|
|(23,[0,1,2,3,4,5,...|     1|       1.0|
|(23,[0,1,2,3,4,5,...|     1|       1.0|
|(23,[0,1,2,3,4,5,...|     1|       1.0|
|(23,[0,1,2,3,4,5,...|     1|       1.0|
|(23,[0,1,2,3,4,5,...|     1|       1.0|
|(23,[0,2,3,4,6,12...|     0|       0.0|
|(23,[0,1,2,3,4,6,...|     0|       0.0|
|(23,[0,1,2,3,4,5,...|     1|       1.0|
|(23,[0,1,2,3,4,7,...|     0|       0.0|
|(23,[0,1,2,3,4,10...|     0|       0.0|
|(23,[0,1,2,3,4,6,...|     0|       0.0|
|(23,[0,1,2,3,4,6,...|     0|       0.0|
|(23,[0,1,2,3,4,9,...|     0|       0.0|
|(23,[0,1,2,3,4,8,...|     0|       0.0|
+--------------------+------+----------+
only showing top

✅ Pipeline(stages=[...])

    stages = steps executed in order

✅ .fit(train_df)

    trains the model on training data

✅ .transform(test_df)

    makes predictions

✅ Why pipelines are important?

    Cleaner code
    
    Easy to reuse
    
    Industry standard for ML workflows